# Shared Knowledge / Private Memory Agents (OpenAI edition)

A notebook version of the **shared-knowledge / private-memory** multi-agent
pattern, using **OpenAI** for both the chat model and the embeddings (so you
only need a single `OPENAI_API_KEY`).

**The idea:** each chat session has its own *private* memory (the raw
conversation, which may contain personal specifics). When you're done with a
session, an LLM-based *consolidation* step distills it into short,
**generalized, anonymized** knowledge snippets and merges them into a single
*shared* knowledge base. Every agent persona and every future session can
draw on that shared knowledge base via retrieval (RAG) — but none of them
can ever see another session's raw private memory.

```
Session A (private, SQLite) --consolidate (LLM, anonymize)--> Shared Knowledge (Chroma, global)
                                                                        |
                                                     retrieved by ANY session/agent
                                                                        v
                                                              Session B (private, SQLite)
```

This notebook builds the whole thing from scratch, then runs a small demo:
chat with a "Coding Tutor" persona, consolidate that session, then chat with
a completely different "Research Assistant" persona and watch it retrieve
the generalized knowledge learned from the *first* persona's session —
without ever seeing that session's actual transcript.

> A Streamlit app version of this same architecture (multi-file project,
> configurable for Anthropic or OpenAI) is available as a companion
> deliverable — see the note at the end of this notebook.


## 1. Install dependencies

In [1]:
%pip install -q langchain langchain-openai langchain-core langgraph langchain-chroma chromadb python-dotenv

Note: you may need to restart the kernel to use updated packages.


## 2. Set up your OpenAI API key

This notebook uses **OpenAI** for everything — the chat model *and* the
embeddings model — so only one key is needed.

Either put it in a `.env` file next to this notebook:

```
OPENAI_API_KEY=sk-...
```

or set it directly in the cell below.

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

# If you'd rather not use a .env file, uncomment and set it directly:
# os.environ["OPENAI_API_KEY"] = "sk-..."

if not os.environ.get("OPENAI_API_KEY"):
    print("⚠️  OPENAI_API_KEY is not set. Set it via a .env file or os.environ before running the chat cells below.")
else:
    print("OPENAI_API_KEY found.")

OPENAI_API_KEY found.


## 3. Config — OpenAI as the provider

Both the chat model and the embeddings model come from OpenAI here:
- Chat: `gpt-4o-mini` (swap for `gpt-4o` or another chat model if you like)
- Embeddings: `text-embedding-3-small` (used only for the shared knowledge vector store)

In [3]:
import json
import re
import time
import uuid
import sqlite3
from pathlib import Path
from typing import TypedDict, List, Dict, Optional

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

LLM_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

DATA_DIR = Path("data")
PRIVATE_DB_PATH = DATA_DIR / "private_memory.db"
SHARED_KB_DIR = DATA_DIR / "shared_knowledge"

DATA_DIR.mkdir(parents=True, exist_ok=True)


def get_llm(temperature: float = 0.4):
    return ChatOpenAI(model=LLM_MODEL, temperature=temperature)


def get_embeddings():
    return OpenAIEmbeddings(model=EMBEDDING_MODEL)

## 4. Private memory — per-session, never shared

A SQLite-backed store keyed by `session_id`. Holds the raw back-and-forth of
one conversation. Nothing in this class ever touches the shared knowledge
store — that's intentional, so private memory code can't leak into the
shared store by accident.

In [4]:
class PrivateMemoryStore:
    def __init__(self, db_path: str = str(PRIVATE_DB_PATH)):
        Path(db_path).parent.mkdir(parents=True, exist_ok=True)
        self.db_path = db_path
        self._init_db()

    def _connect(self):
        conn = sqlite3.connect(self.db_path)
        conn.row_factory = sqlite3.Row
        return conn

    def _init_db(self):
        with self._connect() as conn:
            conn.execute('''
                CREATE TABLE IF NOT EXISTS sessions (
                    session_id TEXT PRIMARY KEY,
                    agent_id TEXT NOT NULL,
                    created_at REAL NOT NULL,
                    consolidated INTEGER DEFAULT 0
                )
            ''')
            conn.execute('''
                CREATE TABLE IF NOT EXISTS messages (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    session_id TEXT NOT NULL,
                    role TEXT NOT NULL,
                    content TEXT NOT NULL,
                    created_at REAL NOT NULL,
                    FOREIGN KEY (session_id) REFERENCES sessions(session_id)
                )
            ''')

    def create_session(self, agent_id: str) -> str:
        session_id = str(uuid.uuid4())
        with self._connect() as conn:
            conn.execute(
                "INSERT INTO sessions (session_id, agent_id, created_at) VALUES (?, ?, ?)",
                (session_id, agent_id, time.time()),
            )
        return session_id

    def delete_session(self, session_id: str):
        with self._connect() as conn:
            conn.execute("DELETE FROM messages WHERE session_id = ?", (session_id,))
            conn.execute("DELETE FROM sessions WHERE session_id = ?", (session_id,))

    def mark_consolidated(self, session_id: str):
        with self._connect() as conn:
            conn.execute("UPDATE sessions SET consolidated = 1 WHERE session_id = ?", (session_id,))

    def list_sessions(self, agent_id: Optional[str] = None) -> List[Dict]:
        with self._connect() as conn:
            if agent_id:
                rows = conn.execute(
                    "SELECT * FROM sessions WHERE agent_id = ? ORDER BY created_at DESC", (agent_id,)
                ).fetchall()
            else:
                rows = conn.execute("SELECT * FROM sessions ORDER BY created_at DESC").fetchall()
        return [dict(r) for r in rows]

    def add_message(self, session_id: str, role: str, content: str):
        with self._connect() as conn:
            conn.execute(
                "INSERT INTO messages (session_id, role, content, created_at) VALUES (?, ?, ?, ?)",
                (session_id, role, content, time.time()),
            )

    def get_messages(self, session_id: str) -> List[Dict]:
        with self._connect() as conn:
            rows = conn.execute(
                "SELECT role, content, created_at FROM messages WHERE session_id = ? ORDER BY id ASC",
                (session_id,),
            ).fetchall()
        return [dict(r) for r in rows]


print("PrivateMemoryStore defined.")

PrivateMemoryStore defined.


## 5. Shared knowledge — global, generalized only

A Chroma vector store holding only distilled, anonymized snippets, retrievable
by any agent/session via semantic search. It must never contain names or
one-off personal specifics — that boundary is enforced by the extractor in
section 7, not by this class itself.

In [5]:
from langchain_chroma import Chroma
from langchain_core.documents import Document


class SharedKnowledgeStore:
    def __init__(self, persist_dir: str = str(SHARED_KB_DIR)):
        self.persist_dir = persist_dir
        self._embeddings = get_embeddings()
        self._store = Chroma(
            collection_name="shared_knowledge",
            embedding_function=self._embeddings,
            persist_directory=self.persist_dir,
        )

    def add_knowledge(self, snippets: List[str], source_agent_id: str, topic: Optional[str] = None):
        docs = [
            Document(
                page_content=snippet.strip(),
                metadata={
                    "source_agent_id": source_agent_id,
                    "topic": topic or "general",
                    "created_at": time.time(),
                    "knowledge_id": str(uuid.uuid4()),
                },
            )
            for snippet in snippets
            if snippet and snippet.strip()
        ]
        if docs:
            self._store.add_documents(docs)

    def query(self, query_text: str, k: int = 4) -> List[Dict]:
        results = self._store.similarity_search_with_score(query_text, k=k)
        return [
            {"content": doc.page_content, "metadata": doc.metadata, "score": score}
            for doc, score in results
        ]

    def all_knowledge(self, limit: int = 200) -> List[Dict]:
        data = self._store.get(limit=limit)
        out = [
            {"content": content, "metadata": metadata}
            for content, metadata in zip(data.get("documents", []), data.get("metadatas", []))
        ]
        out.sort(key=lambda x: x["metadata"].get("created_at", 0), reverse=True)
        return out

    def count(self) -> int:
        return self._store._collection.count()


print("SharedKnowledgeStore defined.")

SharedKnowledgeStore defined.


## 6. Agent personas

Each persona has its own private-memory sessions but reads/writes the **same** shared knowledge base.

In [6]:
AGENTS = {
    "research_assistant": {
        "name": "Research Assistant",
        "system_prompt": (
            "You are a careful research assistant. You help users find, explain, and "
            "reason about factual information. Be precise and flag uncertainty when it exists."
        ),
    },
    "coding_tutor": {
        "name": "Coding Tutor",
        "system_prompt": (
            "You are a patient coding tutor. You help users debug and understand code, "
            "explaining the 'why' behind a fix, not just the fix itself."
        ),
    },
    "wellness_coach": {
        "name": "Wellness Coach",
        "system_prompt": (
            "You are a supportive wellness and productivity coach. You give practical, "
            "grounded advice and avoid generic platitudes."
        ),
    },
}

DEFAULT_AGENT_ID = "research_assistant"
print("Agent personas:", [cfg["name"] for cfg in AGENTS.values()])

Agent personas: ['Research Assistant', 'Coding Tutor', 'Wellness Coach']


## 7. The private → shared bridge: knowledge extraction (anonymization)

This is the **only** code path allowed to move information from private
memory into shared knowledge. It's a deliberate, explicit step (you call it
yourself below), not something that happens automatically on every turn.

The prompt instructs the model to output *only* generalizable knowledge and to
strip anything that identifies or describes the specific individual — names,
contact info, employers, personal dates, one-off facts. If nothing in a
transcript is worth generalizing, it returns an empty list, which is fine.

In [7]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

EXTRACTION_SYSTEM_PROMPT = '''You distill a private conversation transcript into general, reusable \
knowledge for a shared knowledge base that other agents and future sessions can draw on.

Rules (follow strictly):
1. Output ONLY generalizable knowledge: facts about the world, correct answers, useful strategies, \
recurring mistakes and their fixes, domain tips, or stylistic lessons stated as general rules.
2. NEVER include names, emails, phone numbers, addresses, IDs, employers, exact dates tied to a \
person, or any other detail that identifies or describes the specific individual in the conversation.
3. NEVER include one-off personal facts (e.g. "the user's dog is named Max"). If a fact is only \
useful for personalizing replies to this one person, DROP it -- it belongs in private memory, not here.
4. Rephrase everything as an impersonal, standalone statement usable in a different conversation \
with a different person, e.g. "When users ask about X, approach Y works well" rather than \
"I told the user that Y works well".
5. If nothing in the transcript is worth generalizing, return an empty array.
6. Return STRICT JSON only: a JSON array of strings. No prose, no markdown fences, no commentary.

Example output:
["Users asking about async/await bugs in Python often actually have a blocking call inside a \
coroutine; check for that first.", "For budget travel questions, mentioning shoulder-season dates \
is usually the highest-leverage tip."]
'''

_extraction_prompt = ChatPromptTemplate.from_messages([
    ("system", EXTRACTION_SYSTEM_PROMPT),
    ("human", "Transcript:\n\n{transcript}\n\nReturn the JSON array now."),
])


def _format_transcript(messages: List[Dict]) -> str:
    lines = []
    for m in messages:
        role = "User" if m["role"] == "user" else "Assistant"
        lines.append(f"{role}: {m['content']}")
    return "\n".join(lines)


def _parse_json_array(raw: str) -> List[str]:
    cleaned = raw.strip()
    cleaned = re.sub(r"^```(json)?|```$", "", cleaned, flags=re.MULTILINE).strip()
    try:
        data = json.loads(cleaned)
        if isinstance(data, list):
            return [str(x).strip() for x in data if str(x).strip()]
    except json.JSONDecodeError:
        pass
    return []


def extract_generalized_knowledge(messages: List[Dict]) -> List[str]:
    '''Distill a private transcript into anonymized, generalized knowledge snippets.
    Returns an empty list if there's nothing worth generalizing.'''
    if not messages:
        return []
    llm = get_llm(temperature=0.0)
    chain = _extraction_prompt | llm | StrOutputParser()
    raw = chain.invoke({"transcript": _format_transcript(messages)})
    return _parse_json_array(raw)


print("Extractor defined.")

Extractor defined.


## 8. LangGraph agent pipeline

Per turn: retrieve relevant shared knowledge → load this session's private
history → generate a response → persist the new turn to private memory only.

In [8]:
from langgraph.graph import StateGraph, END
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage


class AgentState(TypedDict):
    session_id: str
    agent_id: str
    user_input: str
    private_history: List[Dict]
    shared_context: List[Dict]
    response: str


def build_graph(private_store: "PrivateMemoryStore", shared_store: "SharedKnowledgeStore"):

    def retrieve_shared_knowledge(state: AgentState) -> AgentState:
        state["shared_context"] = shared_store.query(state["user_input"], k=4)
        return state

    def load_private_memory(state: AgentState) -> AgentState:
        state["private_history"] = private_store.get_messages(state["session_id"])
        return state

    def generate_response(state: AgentState) -> AgentState:
        agent_cfg = AGENTS.get(state["agent_id"], AGENTS["research_assistant"])
        llm = get_llm()

        shared_snippets = "\n".join(f"- {r['content']}" for r in state["shared_context"])
        shared_snippets = shared_snippets or "(no relevant shared knowledge yet)"

        system_text = (
            f"{agent_cfg['system_prompt']}\n\n"
            "You have access to generalized knowledge distilled from many past sessions, "
            "across all users and agents. It contains no information about who said what "
            "-- treat it as background expertise, not memory of this specific user. "
            "Use it only if relevant.\n\n"
            f"Relevant shared knowledge:\n{shared_snippets}"
        )

        lc_messages = [SystemMessage(content=system_text)]
        for m in state["private_history"]:
            if m["role"] == "user":
                lc_messages.append(HumanMessage(content=m["content"]))
            else:
                lc_messages.append(AIMessage(content=m["content"]))
        lc_messages.append(HumanMessage(content=state["user_input"]))

        ai_msg = llm.invoke(lc_messages)
        state["response"] = ai_msg.content
        return state

    def persist_turn(state: AgentState) -> AgentState:
        private_store.add_message(state["session_id"], "user", state["user_input"])
        private_store.add_message(state["session_id"], "assistant", state["response"])
        return state

    graph = StateGraph(AgentState)
    graph.add_node("retrieve_shared_knowledge", retrieve_shared_knowledge)
    graph.add_node("load_private_memory", load_private_memory)
    graph.add_node("generate_response", generate_response)
    graph.add_node("persist_turn", persist_turn)

    graph.set_entry_point("retrieve_shared_knowledge")
    graph.add_edge("retrieve_shared_knowledge", "load_private_memory")
    graph.add_edge("load_private_memory", "generate_response")
    graph.add_edge("generate_response", "persist_turn")
    graph.add_edge("persist_turn", END)

    return graph.compile()


print("Graph builder defined.")

Graph builder defined.


## 9. Wire it up

This is where the OpenAI clients actually get created — you need `OPENAI_API_KEY` set for anything past this point.

In [9]:
private_store = PrivateMemoryStore()
shared_store = SharedKnowledgeStore()
graph = build_graph(private_store, shared_store)

print("Stores + graph ready. Shared knowledge currently has", shared_store.count(), "entries.")

Stores + graph ready. Shared knowledge currently has 0 entries.


## 10. Demo: chat, consolidate, then watch knowledge transfer across agents

The flow:
1. Start a **Coding Tutor** session and ask a couple of debugging questions.
2. **End & consolidate** that session — the transcript gets distilled into anonymized knowledge.
3. Start a brand-new **Research Assistant** session (different persona, different private memory)
   and ask a related question — it should retrieve the generalized lesson learned in step 2,
   even though it never saw the Coding Tutor conversation.

In [10]:
def chat(session_id: str, agent_id: str, user_input: str) -> dict:
    result = graph.invoke({
        "session_id": session_id,
        "agent_id": agent_id,
        "user_input": user_input,
        "private_history": [],
        "shared_context": [],
        "response": "",
    })
    print(f"[{AGENTS[agent_id]['name']}] > {user_input}")
    print(result["response"])
    if result["shared_context"]:
        print("\n  (shared knowledge used):")
        for r in result["shared_context"]:
            print("   -", r["content"])
    print("-" * 60)
    return result


# 1. Coding Tutor session
tutor_session = private_store.create_session("coding_tutor")
chat(tutor_session, "coding_tutor", "My Python async function never seems to finish, it just hangs. Any idea why?")
chat(tutor_session, "coding_tutor", "Ah I see, I had a blocking requests.get() call inside the coroutine. That was it, thanks!")

[Coding Tutor] > My Python async function never seems to finish, it just hangs. Any idea why?
When an asynchronous function in Python hangs or never seems to finish, it often indicates that there is an issue with how the asynchronous code is structured or how it interacts with other parts of your program. Here are some common reasons and how to troubleshoot them:

1. **Unawaited Coroutines**: If you have defined an asynchronous function (using `async def`) but never actually awaited it, the function will not execute. Make sure that you are using `await` when calling asynchronous functions.

   ```python
   async def my_async_function():
       # some async code
       await asyncio.sleep(1)  # Example of an awaited call

   # Correct way to call it
   await my_async_function()
   ```

2. **Event Loop Issues**: If you are running your async code outside of an event loop or if the event loop is blocked by synchronous code, it can cause the async function to hang. Make sure that you're ru

{'session_id': '357a3cb1-bd12-4600-9b66-8b86ec8ac9ef',
 'agent_id': 'coding_tutor',
 'user_input': 'Ah I see, I had a blocking requests.get() call inside the coroutine. That was it, thanks!',
 'private_history': [{'role': 'user',
   'content': 'My Python async function never seems to finish, it just hangs. Any idea why?',
   'created_at': 1784225628.1066358},
  {'role': 'assistant',
   'content': "When an asynchronous function in Python hangs or never seems to finish, it often indicates that there is an issue with how the asynchronous code is structured or how it interacts with other parts of your program. Here are some common reasons and how to troubleshoot them:\n\n1. **Unawaited Coroutines**: If you have defined an asynchronous function (using `async def`) but never actually awaited it, the function will not execute. Make sure that you are using `await` when calling asynchronous functions.\n\n   ```python\n   async def my_async_function():\n       # some async code\n       await asy

In [11]:
# 2. Consolidate: distill this session into anonymized, generalized knowledge
messages = private_store.get_messages(tutor_session)
snippets = extract_generalized_knowledge(messages)
shared_store.add_knowledge(snippets, source_agent_id="coding_tutor", topic=AGENTS["coding_tutor"]["name"])
private_store.mark_consolidated(tutor_session)

print(f"Extracted {len(snippets)} generalized snippet(s):")
for s in snippets:
    print(" -", s)

Extracted 7 generalized snippet(s):
 - When an asynchronous function in Python hangs, it may be due to unawaited coroutines; ensure that 'await' is used when calling async functions.
 - Running async code outside of an event loop or blocking the event loop with synchronous code can cause async functions to hang; always run async functions within an event loop.
 - Deadlocks can occur in async functions if they wait for resources that are never released; check for circular dependencies in async calls.
 - Long-running tasks in async functions should yield control back to the event loop; use 'await' on long-running operations or break them into smaller chunks.
 - Uncaught exceptions in async functions can lead to hanging; use try-except blocks to handle exceptions properly.
 - Using 'time.sleep()' in async functions will block the event loop; always use 'await asyncio.sleep()' or other async-compatible methods.
 - For non-blocking HTTP requests in async contexts, use asynchronous libraries

In [12]:
# 3. A completely different persona, completely new private session
research_session = private_store.create_session("research_assistant")
chat(research_session, "research_assistant", "My async Python code seems stuck and never completes. What's a common cause?")

[Research Assistant] > My async Python code seems stuck and never completes. What's a common cause?
A common cause for async Python code getting stuck or never completing is the presence of unawaited coroutines. If you call an async function without using the `await` keyword, the coroutine will not execute, which can lead to the program hanging.

Other potential causes include:

1. **Blocking the Event Loop**: If you are running synchronous code that blocks the event loop or using `time.sleep()` instead of `await asyncio.sleep()`, this will prevent other async tasks from running.

2. **Not Running in an Event Loop**: If you are trying to run async code outside of an event loop, it may not execute as expected. Ensure that your async functions are called within an appropriate event loop.

3. **Long-running Tasks**: If you have long-running operations in your async code, they should yield control back to the event loop. Use `await` on these operations or break them into smaller chunks to 

{'session_id': 'ea345141-3c4e-44e5-a77d-967223e3bed8',
 'agent_id': 'research_assistant',
 'user_input': "My async Python code seems stuck and never completes. What's a common cause?",
 'private_history': [],
 'shared_context': [{'content': "When an asynchronous function in Python hangs, it may be due to unawaited coroutines; ensure that 'await' is used when calling async functions.",
   'metadata': {'topic': 'Coding Tutor',
    'source_agent_id': 'coding_tutor',
    'created_at': 1784225643.8426547,
    'knowledge_id': '60801cf2-0406-4f0b-9158-39b19ef53041'},
   'score': 0.7587205171585083},
  {'content': 'Running async code outside of an event loop or blocking the event loop with synchronous code can cause async functions to hang; always run async functions within an event loop.',
   'metadata': {'created_at': 1784225643.8426547,
    'knowledge_id': 'b588fae9-70a2-4af6-80ef-27154120276b',
    'topic': 'Coding Tutor',
    'source_agent_id': 'coding_tutor'},
   'score': 0.9864564538002

## 11. Inspect the shared knowledge base

This is everything that's been "learned" so far — it should read as generic, reusable advice, with nothing tying it back to the specific conversation above.

In [13]:
for item in shared_store.all_knowledge():
    print(f"- {item['content']}  (from: {item['metadata'].get('topic')})")

- When an asynchronous function in Python hangs, it may be due to unawaited coroutines; ensure that 'await' is used when calling async functions.  (from: Coding Tutor)
- Running async code outside of an event loop or blocking the event loop with synchronous code can cause async functions to hang; always run async functions within an event loop.  (from: Coding Tutor)
- Deadlocks can occur in async functions if they wait for resources that are never released; check for circular dependencies in async calls.  (from: Coding Tutor)
- Long-running tasks in async functions should yield control back to the event loop; use 'await' on long-running operations or break them into smaller chunks.  (from: Coding Tutor)
- Uncaught exceptions in async functions can lead to hanging; use try-except blocks to handle exceptions properly.  (from: Coding Tutor)
- Using 'time.sleep()' in async functions will block the event loop; always use 'await asyncio.sleep()' or other async-compatible methods.  (from: Cod

## 12. Cleanup (optional)

Private memory can be deleted per-session at any time (e.g. "forget this
conversation"). This does **not** touch anything already consolidated into
shared knowledge — consolidation is a one-way, one-time copy of *generalized*
content, not a live link back to the private transcript.

In [ ]:
# Uncomment to delete a session's private memory:
# private_store.delete_session(tutor_session)
# private_store.delete_session(research_session)

print("Sessions on file:", [s["session_id"][:8] for s in private_store.list_sessions()])

## Notes

- **Privacy boundary**: the anonymization step in section 7 is a prompt-level
  safeguard, not a certified privacy guarantee. For production use, consider
  adding a deterministic PII filter (e.g. regex/NER) on extracted snippets
  before they're written to the shared store, or a second "critic" LLM pass
  that vets each snippet.
- **Same architecture, other providers**: swapping `LLM_MODEL`/`EMBEDDING_MODEL`
  and the `get_llm()`/`get_embeddings()` functions for Anthropic (`langchain_anthropic`)
  is enough to change providers — nothing else in this notebook needs to change.
- **Chat UI version**: a Streamlit multi-file project implementing this same
  pattern (with a sidebar to browse the shared knowledge base, switch
  personas, and consolidate sessions with a button) is available as a
  companion deliverable, configurable for either Anthropic or OpenAI.
